# Lab 4: Occupancy Grid Mapping with LiDAR

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fatayer8891-boop/zuj-robotics/blob/main/labs/lab-04-mapping/notebook.ipynb)

---

**Course:** Introduction to Robotics for AI and Data Science (0135343)  
**Instructor:** Dr. Akram Fatayer · [a.fatayer@zuj.edu.jo](mailto:a.fatayer@zuj.edu.jo) · [LinkedIn](https://www.linkedin.com/in/akram-fatayer/)  
**University:** Al-Zaytoonah University of Jordan

---

## Overview

Build occupancy grid maps from LiDAR scans using ray casting and log-odds updates.

> **80/20 Principle:** Focus on understanding the core algorithm in each activity. The implementation details will follow naturally once you grasp the concept.


In [ ]:
# ============================================================
# COLAB ENVIRONMENT SETUP
# ============================================================
# This cell installs the required packages for running this lab
# in Google Colab. If you're running locally, you can skip this.
# ============================================================

import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🔧 Setting up Colab environment...")
    !pip install pybullet numpy matplotlib opencv-python-headless -q
    print("✅ All packages installed!")
    print("⚠️  Note: PyBullet runs in HEADLESS mode on Colab (no 3D GUI).")
    print("   You'll see matplotlib plots instead of the live 3D window.")
    print("   For the full 3D experience, run locally with: conda activate robotics_YourName")
else:
    print("✅ Running locally — full GUI mode available.")


---

## Activity 1: Activity1 Lidar Viz

**File:** `activity1_lidar_viz.py`

Run the code below to complete this activity.


In [ ]:
"""
Week 4 Lab - Activity 1: Visualize LiDAR Data
==============================================

Learning Objectives:
- Understand LiDAR data format (angle, distance)
- Convert polar to Cartesian coordinates
- Visualize point clouds

Author: Manus AI
Course: AI & Robotics
"""

import numpy as np
import matplotlib.pyplot as plt

# ============================================================================
# SIMULATED LIDAR DATA
# ============================================================================

def generate_lidar_scan(robot_x=0.0, robot_y=0.0, robot_theta=0.0):
    """
    Generate a simulated LiDAR scan in a simple environment.
    
    Environment: Rectangular room with obstacles
    - Room: 20m x 20m
    - Obstacles: Two boxes
    
    Parameters:
    -----------
    robot_x, robot_y : float
        Robot position in world frame (meters)
    robot_theta : float
        Robot orientation in world frame (radians)
    
    Returns:
    --------
    angles : np.array
        Scan angles in radians (robot frame)
    ranges : np.array
        Measured distances in meters
    """
    # LiDAR parameters
    num_beams = 360  # 360 beams (1 degree resolution)
    max_range = 15.0  # Maximum range: 15 meters
    noise_std = 0.05  # Measurement noise: 5cm standard deviation
    
    # Scan angles (robot frame): -180° to +180°
    angles = np.linspace(-np.pi, np.pi, num_beams)
    
    # Initialize ranges to max_range
    ranges = np.ones(num_beams) * max_range
    
    # Define environment obstacles (in world frame)
    # Obstacle 1: Box at (5, 3) with size 2m x 2m
    # Obstacle 2: Box at (-4, -5) with size 3m x 1.5m
    obstacles = [
        {'x': 5.0, 'y': 3.0, 'width': 2.0, 'height': 2.0},
        {'x': -4.0, 'y': -5.0, 'width': 3.0, 'height': 1.5},
    ]
    
    # Room boundaries
    room_size = 10.0  # ±10m
    
    # For each beam, ray cast to find nearest obstacle
    for i, angle in enumerate(angles):
        # Beam direction in world frame
        world_angle = robot_theta + angle
        dx = np.cos(world_angle)
        dy = np.sin(world_angle)
        
        min_dist = max_range
        
        # Check room boundaries
        # Right wall (x = room_size)
        if dx > 0:
            t = (room_size - robot_x) / dx
            if t > 0:
                y_hit = robot_y + t * dy
                if abs(y_hit) <= room_size:
                    min_dist = min(min_dist, t)
        
        # Left wall (x = -room_size)
        if dx < 0:
            t = (-room_size - robot_x) / dx
            if t > 0:
                y_hit = robot_y + t * dy
                if abs(y_hit) <= room_size:
                    min_dist = min(min_dist, t)
        
        # Top wall (y = room_size)
        if dy > 0:
            t = (room_size - robot_y) / dy
            if t > 0:
                x_hit = robot_x + t * dx
                if abs(x_hit) <= room_size:
                    min_dist = min(min_dist, t)
        
        # Bottom wall (y = -room_size)
        if dy < 0:
            t = (-room_size - robot_y) / dy
            if t > 0:
                x_hit = robot_x + t * dx
                if abs(x_hit) <= room_size:
                    min_dist = min(min_dist, t)
        
        # Check obstacles (simplified: treat as rectangles)
        for obs in obstacles:
            # Check four edges of the obstacle
            x_min = obs['x'] - obs['width'] / 2
            x_max = obs['x'] + obs['width'] / 2
            y_min = obs['y'] - obs['height'] / 2
            y_max = obs['y'] + obs['height'] / 2
            
            # Right edge
            if dx > 0:
                t = (x_max - robot_x) / dx
                if t > 0:
                    y_hit = robot_y + t * dy
                    if y_min <= y_hit <= y_max:
                        min_dist = min(min_dist, t)
            
            # Left edge
            if dx < 0:
                t = (x_min - robot_x) / dx
                if t > 0:
                    y_hit = robot_y + t * dy
                    if y_min <= y_hit <= y_max:
                        min_dist = min(min_dist, t)
            
            # Top edge
            if dy > 0:
                t = (y_max - robot_y) / dy
                if t > 0:
                    x_hit = robot_x + t * dx
                    if x_min <= x_hit <= x_max:
                        min_dist = min(min_dist, t)
            
            # Bottom edge
            if dy < 0:
                t = (y_min - robot_y) / dy
                if t > 0:
                    x_hit = robot_x + t * dx
                    if x_min <= x_hit <= x_max:
                        min_dist = min(min_dist, t)
        
        ranges[i] = min_dist
    
    # Add measurement noise
    ranges += np.random.normal(0, noise_std, num_beams)
    ranges = np.clip(ranges, 0.1, max_range)  # Clip to valid range
    
    return angles, ranges


def polar_to_cartesian(angles, ranges):
    """
    Convert polar coordinates (angle, range) to Cartesian (x, y).
    
    Parameters:
    -----------
    angles : np.array
        Angles in radians
    ranges : np.array
        Distances in meters
    
    Returns:
    --------
    x, y : np.array
        Cartesian coordinates in meters
    """
    x = ranges * np.cos(angles)
    y = ranges * np.sin(angles)
    return x, y


# ============================================================================
# VISUALIZATION
# ============================================================================

def visualize_lidar_scan(angles, ranges, robot_x=0.0, robot_y=0.0):
    """
    Visualize LiDAR scan as a point cloud.
    
    Parameters:
    -----------
    angles : np.array
        Scan angles in radians
    ranges : np.array
        Measured distances in meters
    robot_x, robot_y : float
        Robot position for plotting
    """
    # Convert to Cartesian
    x, y = polar_to_cartesian(angles, ranges)
    
    # Offset by robot position
    x += robot_x
    y += robot_y
    
    # Create figure
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # Plot 1: Polar plot (range vs angle)
    ax1.plot(np.degrees(angles), ranges, 'b-', linewidth=0.5)
    ax1.scatter(np.degrees(angles), ranges, c='blue', s=1, alpha=0.5)
    ax1.set_xlabel('Angle (degrees)', fontsize=12)
    ax1.set_ylabel('Range (meters)', fontsize=12)
    ax1.set_title('LiDAR Scan: Range vs Angle', fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    ax1.set_xlim(-180, 180)
    
    # Plot 2: Cartesian point cloud
    ax2.scatter(x, y, c='red', s=2, alpha=0.6, label='LiDAR points')
    ax2.plot(robot_x, robot_y, 'go', markersize=10, label='Robot')
    ax2.set_xlabel('X Position (m)', fontsize=12)
    ax2.set_ylabel('Y Position (m)', fontsize=12)
    ax2.set_title('Point Cloud Visualization', fontsize=14, fontweight='bold')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    ax2.axis('equal')
    ax2.set_xlim(-12, 12)
    ax2.set_ylim(-12, 12)
    
    plt.tight_layout()
    plt.savefig('activity1_lidar_viz.png', dpi=150, bbox_inches='tight')
    print("✓ Saved: activity1_lidar_viz.png")
    plt.show()


# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":
    print("=" * 60)
    print("Activity 1: Visualize LiDAR Data")
    print("=" * 60)
    
    # Robot pose
    robot_x = 0.0
    robot_y = 0.0
    robot_theta = 0.0  # Facing right (+x direction)
    
    print(f"\nRobot pose: ({robot_x:.1f}, {robot_y:.1f}), θ = {np.degrees(robot_theta):.1f}°")
    
    # Generate LiDAR scan
    print("\nGenerating simulated LiDAR scan...")
    angles, ranges = generate_lidar_scan(robot_x, robot_y, robot_theta)
    
    print(f"  Number of beams: {len(angles)}")
    print(f"  Angular resolution: {np.degrees(angles[1] - angles[0]):.2f}°")
    print(f"  Range: {ranges.min():.2f}m to {ranges.max():.2f}m")
    
    # Visualize
    print("\nVisualizing LiDAR scan...")
    visualize_lidar_scan(angles, ranges, robot_x, robot_y)
    
    print("\n" + "=" * 60)
    print("Activity 1 Complete!")
    print("=" * 60)
    print("\nKey Observations:")
    print("  • LiDAR provides 360° coverage around the robot")
    print("  • Point cloud shows room boundaries and obstacles")
    print("  • Polar representation shows range vs angle")
    print("  • Cartesian representation shows spatial layout")



---

## Activity 2: Activity2 Basic Grid

**File:** `activity2_basic_grid.py`

Run the code below to complete this activity.


In [ ]:
"""
Week 4 Lab - Activity 2: Basic Occupancy Grid
==============================================

Learning Objectives:
- Implement occupancy grid data structure
- Convert world coordinates to grid indices
- Mark occupied cells from LiDAR endpoints
- Visualize grid as grayscale image

Author: Manus AI
Course: AI & Robotics
"""

import numpy as np
import matplotlib.pyplot as plt
from activity1_lidar_viz import generate_lidar_scan, polar_to_cartesian

# ============================================================================
# OCCUPANCY GRID CLASS
# ============================================================================

class OccupancyGrid:
    """
    2D Occupancy Grid for robot mapping.
    
    Attributes:
    -----------
    width, height : int
        Grid dimensions in cells
    resolution : float
        Cell size in meters
    origin_x, origin_y : float
        World coordinates of grid[0][0]
    grid : np.array
        2D array storing occupancy probabilities [0, 1]
    """
    
    def __init__(self, width, height, resolution, origin_x, origin_y):
        """
        Initialize empty occupancy grid.
        
        Parameters:
        -----------
        width, height : int
            Grid dimensions in cells
        resolution : float
            Cell size in meters (e.g., 0.1 = 10cm)
        origin_x, origin_y : float
            World coordinates of grid[0][0] (bottom-left corner)
        """
        self.width = width
        self.height = height
        self.resolution = resolution
        self.origin_x = origin_x
        self.origin_y = origin_y
        
        # Initialize grid: 0.5 = unknown
        self.grid = np.ones((height, width)) * 0.5
        
        print(f"Initialized grid: {width}x{height} cells")
        print(f"  Resolution: {resolution}m per cell")
        print(f"  Coverage: {width*resolution}m x {height*resolution}m")
        print(f"  Origin: ({origin_x}, {origin_y})")
    
    def world_to_grid(self, x, y):
        """
        Convert world coordinates to grid indices.
        
        Parameters:
        -----------
        x, y : float or np.array
            World coordinates in meters
        
        Returns:
        --------
        gx, gy : int or np.array
            Grid indices (column, row)
        """
        gx = ((x - self.origin_x) / self.resolution).astype(int)
        gy = ((y - self.origin_y) / self.resolution).astype(int)
        return gx, gy
    
    def grid_to_world(self, gx, gy):
        """
        Convert grid indices to world coordinates (cell center).
        
        Parameters:
        -----------
        gx, gy : int or np.array
            Grid indices (column, row)
        
        Returns:
        --------
        x, y : float or np.array
            World coordinates in meters
        """
        x = self.origin_x + (gx + 0.5) * self.resolution
        y = self.origin_y + (gy + 0.5) * self.resolution
        return x, y
    
    def is_valid(self, gx, gy):
        """
        Check if grid indices are within bounds.
        
        Parameters:
        -----------
        gx, gy : int or np.array
            Grid indices
        
        Returns:
        --------
        valid : bool or np.array
            True if indices are within grid bounds
        """
        return (gx >= 0) & (gx < self.width) & (gy >= 0) & (gy < self.height)
    
    def mark_occupied(self, x, y):
        """
        Mark a cell as occupied (probability = 1.0).
        
        Parameters:
        -----------
        x, y : float or np.array
            World coordinates in meters
        """
        gx, gy = self.world_to_grid(x, y)
        
        # Handle arrays
        if isinstance(gx, np.ndarray):
            valid = self.is_valid(gx, gy)
            gx = gx[valid]
            gy = gy[valid]
        else:
            if not self.is_valid(gx, gy):
                return
        
        # Mark as occupied
        self.grid[gy, gx] = 1.0
    
    def visualize(self, robot_x=None, robot_y=None, title="Occupancy Grid"):
        """
        Visualize occupancy grid as grayscale image.
        
        Parameters:
        -----------
        robot_x, robot_y : float, optional
            Robot position to plot
        title : str
            Plot title
        """
        fig, ax = plt.subplots(figsize=(10, 10))
        
        # Display grid (flip vertically for correct orientation)
        ax.imshow(np.flipud(self.grid), cmap='gray', origin='lower', 
                  extent=[self.origin_x, self.origin_x + self.width * self.resolution,
                          self.origin_y, self.origin_y + self.height * self.resolution],
                  vmin=0, vmax=1)
        
        # Plot robot position
        if robot_x is not None and robot_y is not None:
            ax.plot(robot_x, robot_y, 'ro', markersize=10, label='Robot')
            ax.legend()
        
        ax.set_xlabel('X Position (m)', fontsize=12)
        ax.set_ylabel('Y Position (m)', fontsize=12)
        ax.set_title(title, fontsize=14, fontweight='bold')
        ax.grid(True, alpha=0.3)
        ax.axis('equal')
        
        # Add colorbar
        cbar = plt.colorbar(ax.images[0], ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label('Occupancy Probability', fontsize=12)
        
        plt.tight_layout()
        return fig, ax


# ============================================================================
# MAPPING FUNCTION
# ============================================================================

def build_basic_map(robot_poses, num_scans=10):
    """
    Build a basic occupancy grid by marking LiDAR endpoints as occupied.
    
    Parameters:
    -----------
    robot_poses : list of tuples
        List of (x, y, theta) robot poses
    num_scans : int
        Number of scans to process
    
    Returns:
    --------
    grid : OccupancyGrid
        Resulting occupancy grid
    """
    # Initialize grid
    grid = OccupancyGrid(
        width=200,      # 200 cells
        height=200,     # 200 cells
        resolution=0.1,  # 10cm per cell
        origin_x=-10.0,  # Grid covers -10m to +10m
        origin_y=-10.0
    )
    
    print(f"\nProcessing {num_scans} scans...")
    
    for i, (robot_x, robot_y, robot_theta) in enumerate(robot_poses[:num_scans]):
        # Generate LiDAR scan
        angles, ranges = generate_lidar_scan(robot_x, robot_y, robot_theta)
        
        # Convert to Cartesian (robot frame)
        x_robot, y_robot = polar_to_cartesian(angles, ranges)
        
        # Transform to world frame
        cos_theta = np.cos(robot_theta)
        sin_theta = np.sin(robot_theta)
        x_world = robot_x + x_robot * cos_theta - y_robot * sin_theta
        y_world = robot_y + x_robot * sin_theta + y_robot * cos_theta
        
        # Mark endpoints as occupied
        grid.mark_occupied(x_world, y_world)
        
        if (i + 1) % 5 == 0:
            print(f"  Processed {i + 1}/{num_scans} scans")
    
    print("✓ Mapping complete!")
    return grid


# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":
    print("=" * 60)
    print("Activity 2: Basic Occupancy Grid")
    print("=" * 60)
    
    # Define robot trajectory (stationary for now)
    robot_poses = [
        (0.0, 0.0, 0.0),           # Scan 1: Center, facing right
        (0.0, 0.0, np.pi/4),       # Scan 2: Center, rotated 45°
        (0.0, 0.0, np.pi/2),       # Scan 3: Center, facing up
        (0.0, 0.0, 3*np.pi/4),     # Scan 4: Center, rotated 135°
        (0.0, 0.0, np.pi),         # Scan 5: Center, facing left
        (2.0, 0.0, 0.0),           # Scan 6: Moved right
        (-2.0, 0.0, np.pi),        # Scan 7: Moved left
        (0.0, 2.0, -np.pi/2),      # Scan 8: Moved up
        (0.0, -2.0, np.pi/2),      # Scan 9: Moved down
        (3.0, 3.0, -3*np.pi/4),    # Scan 10: Corner
    ]
    
    # Build map
    grid = build_basic_map(robot_poses, num_scans=10)
    
    # Visualize
    print("\nVisualizing occupancy grid...")
    fig, ax = grid.visualize(robot_x=0.0, robot_y=0.0, 
                              title="Basic Occupancy Grid (Endpoints Only)")
    plt.savefig('activity2_basic_grid.png', dpi=150, bbox_inches='tight')
    print("✓ Saved: activity2_basic_grid.png")
    plt.show()
    
    # Statistics
    occupied_cells = np.sum(grid.grid == 1.0)
    unknown_cells = np.sum(grid.grid == 0.5)
    total_cells = grid.width * grid.height
    
    print("\n" + "=" * 60)
    print("Grid Statistics:")
    print("=" * 60)
    print(f"  Total cells: {total_cells}")
    print(f"  Occupied cells: {occupied_cells} ({100*occupied_cells/total_cells:.2f}%)")
    print(f"  Unknown cells: {unknown_cells} ({100*unknown_cells/total_cells:.2f}%)")
    print(f"  Free cells: 0 (not marked yet)")
    
    print("\n" + "=" * 60)
    print("Activity 2 Complete!")
    print("=" * 60)
    print("\nKey Observations:")
    print("  • Endpoints marked as occupied (black)")
    print("  • Most of grid remains unknown (gray)")
    print("  • No free space marked yet (need ray casting)")
    print("  • Next: Activity 3 will add inverse sensor model")



---

## Activity 3: Warehouse Environment

**File:** `warehouse_environment.py`

Run the code below to complete this activity.


In [ ]:
"""
Warehouse Environment for Autonomous Robot Project

This module provides a simulated warehouse environment for the course project.
Students will extend this base environment throughout the semester.

Course: Introduction to Robotics for AI and Data Science
Institution: Al-Zaytoonah University of Jordan
"""

import pybullet as p
import pybullet_data
import numpy as np
import time
import os

class WarehouseEnvironment:
    """
    Simulated warehouse environment with shelves, boxes, and obstacles.
    
    This class manages the PyBullet simulation, including:
    - Loading the warehouse layout
    - Spawning objects (boxes, obstacles)
    - Managing the robot
    - Providing sensor interfaces
    """
    
    def __init__(self, gui=True, warehouse_size=(10, 10)):
        """
        Initialize the warehouse environment.
        
        Args:
            gui (bool): If True, show 3D visualization window
            warehouse_size (tuple): (width, length) of warehouse in meters
        """
        self.gui = gui
        self.warehouse_size = warehouse_size
        self.robot_id = None
        self.objects = {}  # Dictionary to store object IDs
        self.target_items = []  # List of items to retrieve
        
        # Connect to physics server
        if gui:
            self.client = p.connect(p.DIRECT if IN_COLAB else p.GUI)
            p.configureDebugVisualizer(p.COV_ENABLE_GUI, 0)  # Disable GUI controls
        else:
            self.client = p.connect(p.DIRECT)
        
        # Set up simulation
        p.setAdditionalSearchPath(pybullet_data.getDataPath())
        p.setGravity(0, 0, -9.81)
        p.setTimeStep(1./240.)
        
        # Load environment
        self._load_environment()
        
        print(f"Warehouse environment initialized: {warehouse_size[0]}m x {warehouse_size[1]}m")
    
    def _load_environment(self):
        """Load the warehouse layout (floor, walls, shelves)."""
        # Load ground plane
        self.plane_id = p.loadURDF("plane.urdf")
        
        # Create warehouse boundaries (walls)
        self._create_walls()
        
        # Create shelving units
        self._create_shelves()
        
        print("Warehouse layout loaded")
    
    def _create_walls(self):
        """Create walls around the warehouse perimeter."""
        width, length = self.warehouse_size
        wall_height = 2.0
        wall_thickness = 0.1
        
        # Wall positions: [x, y, z]
        # North wall
        self._create_wall([0, length/2, wall_height/2], 
                         [width/2, wall_thickness/2, wall_height/2])
        # South wall
        self._create_wall([0, -length/2, wall_height/2], 
                         [width/2, wall_thickness/2, wall_height/2])
        # East wall
        self._create_wall([width/2, 0, wall_height/2], 
                         [wall_thickness/2, length/2, wall_height/2])
        # West wall
        self._create_wall([-width/2, 0, wall_height/2], 
                         [wall_thickness/2, length/2, wall_height/2])
    
    def _create_wall(self, position, half_extents):
        """Create a single wall using a box collision shape."""
        collision_shape = p.createCollisionShape(p.GEOM_BOX, halfExtents=half_extents)
        visual_shape = p.createVisualShape(p.GEOM_BOX, halfExtents=half_extents,
                                          rgbaColor=[0.7, 0.7, 0.7, 1.0])
        wall_id = p.createMultiBody(baseMass=0,  # Static object
                                    baseCollisionShapeIndex=collision_shape,
                                    baseVisualShapeIndex=visual_shape,
                                    basePosition=position)
        return wall_id
    
    def _create_shelves(self):
        """Create shelving units in the warehouse."""
        # Simple shelves: vertical posts with horizontal platforms
        shelf_positions = [
            [2, 2, 0],
            [2, -2, 0],
            [-2, 2, 0],
            [-2, -2, 0]
        ]
        
        for pos in shelf_positions:
            self._create_shelf(pos)
    
    def _create_shelf(self, position):
        """Create a single shelf unit."""
        # Vertical post
        post_height = 1.5
        post_radius = 0.10
        collision_shape = p.createCollisionShape(p.GEOM_CYLINDER, 
                                                 radius=post_radius, 
                                                 height=post_height)
        visual_shape = p.createVisualShape(p.GEOM_CYLINDER,
                                          radius=post_radius,
                                          length=post_height,
                                          rgbaColor=[0.6, 0.4, 0.2, 1.0])
        
        post_pos = [position[0], position[1], post_height/2]
        shelf_id = p.createMultiBody(baseMass=0,
                                    baseCollisionShapeIndex=collision_shape,
                                    baseVisualShapeIndex=visual_shape,
                                    basePosition=post_pos)
        return shelf_id
    
    def spawn_robot(self, robot_type="husky", position=None):
        """
        Spawn a robot in the warehouse.
        
        Args:
            robot_type (str): Type of robot ("husky", "r2d2", etc.)
            position (list): [x, y, z] starting position, or None for center
        
        Returns:
            int: Robot ID
        """
        if position is None:
            position = [0, 0, 0.1]
        
        orientation = p.getQuaternionFromEuler([0, 0, 0])
        
        if robot_type == "husky":
            self.robot_id = p.loadURDF("husky/husky.urdf", position, orientation)
        elif robot_type == "r2d2":
            self.robot_id = p.loadURDF("r2d2.urdf", position, orientation)
        else:
            raise ValueError(f"Unknown robot type: {robot_type}")
        
        print(f"Robot spawned: {robot_type} at position {position}")
        return self.robot_id
    
    def spawn_box(self, position, color="red", size=0.5, name=None):
        """
        Spawn a colored box (item to retrieve).
        
        Args:
            position (list): [x, y, z] position
            color (str): Color name ("red", "blue", "green", "yellow")
            size (float): Box size in meters
            name (str): Optional name for the box
        
        Returns:
            int: Box ID
        """
        # Color mapping
        colors = {
            "red": [1, 0, 0, 1],
            "blue": [0, 0, 1, 1],
            "green": [0, 1, 0, 1],
            "yellow": [1, 1, 0, 1],
            "orange": [1, 0.5, 0, 1],
            "purple": [0.5, 0, 0.5, 1]
        }
        
        rgba = colors.get(color, [0.5, 0.5, 0.5, 1])
        
        # Create box
        half_extents = [size/2, size/2, size/2]
        collision_shape = p.createCollisionShape(p.GEOM_BOX, halfExtents=half_extents)
        visual_shape = p.createVisualShape(p.GEOM_BOX, halfExtents=half_extents,
                                          rgbaColor=rgba)
        
        box_id = p.createMultiBody(baseMass=0.1,  # Light weight
                                   baseCollisionShapeIndex=collision_shape,
                                   baseVisualShapeIndex=visual_shape,
                                   basePosition=position)
        
        # Store box info
        if name is None:
            name = f"{color}_box_{len(self.objects)}"
        
        self.objects[name] = {
            'id': box_id,
            'type': 'box',
            'color': color,
            'position': position,
            'size': size
        }
        
        print(f"Box spawned: {name} ({color}) at position {position}")
        return box_id
    
    def spawn_random_boxes(self, num_boxes=3):
        """
        Spawn random colored boxes in the warehouse.
        
        Args:
            num_boxes (int): Number of boxes to spawn
        """
        colors = ["red", "blue", "green", "yellow", "orange"]
        width, length = self.warehouse_size
        
        for i in range(num_boxes):
            # Random position (avoid center where robot spawns)
            x = np.random.uniform(-width/3, width/3)
            y = np.random.uniform(-length/3, length/3)
            z = 0.1  # Just above ground
            
            # Ensure not too close to center
            if abs(x) < 1.0 and abs(y) < 1.0:
                x += 1.5 * np.sign(x) if x != 0 else 1.5
            
            color = colors[i % len(colors)]
            self.spawn_box([x, y, z], color=color, name=f"box_{i}")
    
    def get_robot_state(self):
        """
        Get current robot state (position, orientation, velocity).
        
        Returns:
            dict: Robot state information
        """
        if self.robot_id is None:
            return None
        
        position, orientation = p.getBasePositionAndOrientation(self.robot_id)
        linear_vel, angular_vel = p.getBaseVelocity(self.robot_id)
        euler = p.getEulerFromQuaternion(orientation)
        
        return {
            'position': position,
            'orientation': orientation,
            'euler': euler,  # (roll, pitch, yaw)
            'linear_velocity': linear_vel,
            'angular_velocity': angular_vel
        }
    
    def move_robot(self, linear_velocity, angular_velocity):
        """
        Command robot to move with specified velocities.
        
        Args:
            linear_velocity (float): Forward speed in m/s
            angular_velocity (float): Turning rate in rad/s
        """
        if self.robot_id is None:
            print("No robot spawned!")
            return
        
        # For Husky robot: control wheel joints directly
        # Husky has 4 wheels: front_left, front_right, rear_left, rear_right
        # Wheel radius: ~0.165m, wheelbase: ~0.555m
        wheel_radius = 0.165
        wheelbase = 0.555
        
        # Calculate wheel velocities for differential drive
        # v_left = v - (w * L / 2)
        # v_right = v + (w * L / 2)
        v_left = linear_velocity - (angular_velocity * wheelbase / 2.0)
        v_right = linear_velocity + (angular_velocity * wheelbase / 2.0)
        
        # Convert linear wheel velocity to angular velocity (rad/s)
        wheel_vel_left = v_left / wheel_radius
        wheel_vel_right = v_right / wheel_radius
        
        # Get number of joints
        num_joints = p.getNumJoints(self.robot_id)
        
        # Find wheel joints and set their velocities
        for joint_index in range(num_joints):
            joint_info = p.getJointInfo(self.robot_id, joint_index)
            joint_name = joint_info[1].decode('utf-8')
            
            # Set velocity for each wheel
            if 'left' in joint_name.lower() and 'wheel' in joint_name.lower():
                p.setJointMotorControl2(self.robot_id, joint_index,
                                       p.VELOCITY_CONTROL,
                                       targetVelocity=wheel_vel_left,
                                       force=20)
            elif 'right' in joint_name.lower() and 'wheel' in joint_name.lower():
                p.setJointMotorControl2(self.robot_id, joint_index,
                                       p.VELOCITY_CONTROL,
                                       targetVelocity=wheel_vel_right,
                                       force=20)
    
    def get_object_positions(self):
        """
        Get positions of all objects in the warehouse.
        
        Returns:
            dict: Object names mapped to positions
        """
        positions = {}
        for name, obj_info in self.objects.items():
            pos, _ = p.getBasePositionAndOrientation(obj_info['id'])
            positions[name] = pos
        return positions
    
    def step(self, sleep_time=None):
        """
        Step the simulation forward by one time step.
        
        Args:
            sleep_time (float): Optional sleep time for real-time visualization
        """
        p.stepSimulation()
        if sleep_time is not None:
            time.sleep(sleep_time)
    
    def run(self, duration=5.0, real_time=True):
        """
        Run simulation for a specified duration.
        
        Args:
            duration (float): Simulation duration in seconds
            real_time (bool): If True, run at real-time speed
        """
        steps = int(duration * 240)  # 240 Hz
        sleep_time = 1./240. if real_time else None
        
        for _ in range(steps):
            self.step(sleep_time)
    
    def close(self):
        """Disconnect from physics server and cleanup."""
        p.disconnect()
        print("Warehouse environment closed")
    
    def reset(self):
        """Reset the simulation to initial state."""
        p.resetSimulation()
        self.objects = {}
        self.target_items = []
        self._load_environment()
        print("Environment reset")


# Example usage
if __name__ == "__main__":
    # Create warehouse environment
    env = WarehouseEnvironment(gui=True, warehouse_size=(10, 10))
    
    # Spawn robot
    env.spawn_robot(robot_type="husky", position=[0, 0, 0.1])
    
    # Spawn some colored boxes
    env.spawn_random_boxes(num_boxes=5)
    
    # Run simulation for 10 seconds
    print("\nRunning simulation for 10 seconds...")
    env.run(duration=10.0, real_time=True)
    
    # Get final positions
    print("\nFinal object positions:")
    positions = env.get_object_positions()
    for name, pos in positions.items():
        print(f"  {name}: ({pos[0]:.2f}, {pos[1]:.2f}, {pos[2]:.2f})")
    
    # Cleanup
    env.close()



---

## Activity 4: Activity3 Warehouse Mapping

**File:** `activity3_warehouse_mapping.py`

Run the code below to complete this activity.


In [ ]:
"""
Activity 3: Real-Time Warehouse Mapping with PyBullet

This activity integrates everything you've learned:
- LiDAR sensor simulation (Activity 1)
- Occupancy grid mapping (Activity 2)
- PyBullet warehouse environment (Week 2)

Your robot will explore the warehouse and build an occupancy grid map in real-time.

Course: Introduction to Robotics for AI and Data Science
Week 4: Distance Sensing and Occupancy Grid Mapping
"""

import pybullet as p
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import time
from warehouse_environment import WarehouseEnvironment

class LiDARSensor:
    """
    Simulated 2D LiDAR sensor using PyBullet ray casting.
    
    This sensor shoots rays in a 360-degree pattern and measures
    distances to obstacles.
    """
    
    def __init__(self, num_rays=360, max_range=10.0, min_range=0.01):
        """
        Initialize LiDAR sensor.
        
        Args:
            num_rays (int): Number of laser rays (angular resolution)
            max_range (float): Maximum detection range in meters
            min_range (float): Minimum detection range in meters
        """
        self.num_rays = num_rays
        self.max_range = max_range
        self.min_range = min_range
        self.angles = np.linspace(0, 2*np.pi, num_rays, endpoint=False)
        
    def scan(self, robot_id, sensor_height=0.5):
        """
        Perform a LiDAR scan from the robot's current position.
        
        Args:
            robot_id (int): PyBullet robot ID
            sensor_height (float): Height of sensor above robot base
            
        Returns:
            tuple: (ranges, angles) where ranges is array of distances
        """
        # Get robot position and orientation
        pos, orn = p.getBasePositionAndOrientation(robot_id)
        euler = p.getEulerFromQuaternion(orn)
        yaw = euler[2]  # Robot's heading angle
        
        # Sensor position (on top of robot)
        sensor_pos = [pos[0], pos[1], pos[2] + sensor_height]
        
        # Prepare ray casting
        ray_from = []
        ray_to = []
        
        for angle in self.angles:
            # Global angle = robot yaw + sensor angle
            global_angle = yaw + angle
            
            # Calculate ray endpoints
            from_point = sensor_pos
            to_point = [
                sensor_pos[0] + self.max_range * np.cos(global_angle),
                sensor_pos[1] + self.max_range * np.sin(global_angle),
                sensor_pos[2]
            ]
            
            ray_from.append(from_point)
            ray_to.append(to_point)
        
        # Perform batch ray casting (efficient!)
        results = p.rayTestBatch(ray_from, ray_to)
        
        # Extract ranges
        ranges = []
        for i, result in enumerate(results):
            object_id, link_index, hit_fraction, hit_position, hit_normal = result
            
            if object_id == -1:  # No hit
                ranges.append(self.max_range)
            else:
                # Calculate actual distance
                distance = hit_fraction * self.max_range
                ranges.append(max(distance, self.min_range))
        
        return np.array(ranges), self.angles


class OccupancyGridMapper:
    """
    Real-time occupancy grid mapper using log-odds representation.
    
    This class builds a 2D occupancy grid map from LiDAR scans.
    """
    
    def __init__(self, width=20.0, height=20.0, resolution=0.01):
        """
        Initialize occupancy grid.
        
        Args:
            width (float): Map width in meters
            height (float): Map height in meters
            resolution (float): Grid cell size in meters
        """
        self.width = width
        self.height = height
        self.resolution = resolution
        
        # Calculate grid dimensions
        self.grid_width = int(width / resolution)
        self.grid_height = int(height / resolution)
        
        # Initialize log-odds grid (0 = unknown)
        self.log_odds = np.zeros((self.grid_height, self.grid_width))
        
        # Log-odds update values
        self.l_occ = 0.85   # Log-odds for occupied
        self.l_free = -0.4  # Log-odds for free
        
        # Clamping limits
        self.l_min = -5.0
        self.l_max = 5.0
        
        # Map origin (center of grid)
        self.origin_x = width / 2.0
        self.origin_y = height / 2.0
        
        print(f"Occupancy grid initialized: {self.grid_width}x{self.grid_height} cells")
        print(f"Resolution: {resolution}m, Coverage: {width}m x {height}m")
    
    def world_to_grid(self, x, y):
        """
        Convert world coordinates to grid indices.
        
        Args:
            x, y (float): World coordinates in meters
            
        Returns:
            tuple: (row, col) grid indices, or None if out of bounds
        """
        # Transform to grid coordinates
        grid_x = int((x + self.origin_x) / self.resolution)
        grid_y = int((y + self.origin_y) / self.resolution)
        
        # Check bounds
        if 0 <= grid_x < self.grid_width and 0 <= grid_y < self.grid_height:
            return (grid_y, grid_x)  # Note: row=y, col=x
        return None
    
    def bresenham_line(self, x0, y0, x1, y1):
        """
        Bresenham's line algorithm to get all cells along a line.
        
        Args:
            x0, y0: Start grid coordinates
            x1, y1: End grid coordinates
            
        Returns:
            list: List of (row, col) tuples along the line
        """
        cells = []
        
        dx = abs(x1 - x0)
        dy = abs(y1 - y0)
        
        sx = 1 if x0 < x1 else -1
        sy = 1 if y0 < y1 else -1
        
        err = dx - dy
        
        x, y = x0, y0
        
        while True:
            cells.append((y, x))  # (row, col)
            
            if x == x1 and y == y1:
                break
            
            e2 = 2 * err
            
            if e2 > -dy:
                err -= dy
                x += sx
            
            if e2 < dx:
                err += dx
                y += sy
        
        return cells
    
    def update_from_scan(self, robot_x, robot_y, ranges, angles):
        """
        Update occupancy grid from a LiDAR scan.
        
        Args:
            robot_x, robot_y (float): Robot position in world coordinates
            ranges (array): Array of range measurements
            angles (array): Array of corresponding angles (global frame)
        """
        # Get robot grid position
        robot_grid = self.world_to_grid(robot_x, robot_y)
        if robot_grid is None:
            return  # Robot outside map
        
        robot_row, robot_col = robot_grid
        
        # Process each ray
        for r, angle in zip(ranges, angles):
            # Skip invalid measurements
            if r >= 9.9:  # Max range (no obstacle detected)
                continue
            
            # Calculate endpoint in world coordinates
            end_x = robot_x + r * np.cos(angle)
            end_y = robot_y + r * np.sin(angle)
            
            # Convert to grid coordinates
            end_grid = self.world_to_grid(end_x, end_y)
            if end_grid is None:
                continue  # Endpoint outside map
            
            end_row, end_col = end_grid
            
            # Get all cells along the ray using Bresenham
            ray_cells = self.bresenham_line(robot_col, robot_row, end_col, end_row)
            
            # Update cells along the ray
            for i, (row, col) in enumerate(ray_cells):
                # Check bounds
                if not (0 <= row < self.grid_height and 0 <= col < self.grid_width):
                    continue
                
                if i == len(ray_cells) - 1:
                    # Last cell = occupied (obstacle detected)
                    self.log_odds[row, col] += self.l_occ
                else:
                    # Intermediate cells = free space
                    self.log_odds[row, col] += self.l_free
                
                # Clamp to limits
                self.log_odds[row, col] = np.clip(self.log_odds[row, col], 
                                                   self.l_min, self.l_max)
    
    def get_probability_grid(self):
        """
        Convert log-odds to probability (for visualization).
        
        Returns:
            array: Probability grid (0 to 1)
        """
        # Convert log-odds to probability: p = 1 - 1/(1 + exp(l))
        prob = 1.0 - 1.0 / (1.0 + np.exp(self.log_odds))
        return prob
    
    def get_occupancy_grid(self, threshold=0.5):
        """
        Get binary occupancy grid (occupied/free/unknown).
        
        Args:
            threshold (float): Probability threshold for occupied
            
        Returns:
            array: Grid with values: -1 (unknown), 0 (free), 1 (occupied)
        """
        prob = self.get_probability_grid()
        
        grid = np.full_like(prob, -1)  # Unknown
        grid[prob < 0.4] = 0   # Free
        grid[prob > 0.6] = 1   # Occupied
        
        return grid


class WarehouseMapper:
    """
    Main class for warehouse mapping demo.
    
    Integrates PyBullet simulation, LiDAR sensor, and occupancy grid mapping.
    """
    
    def __init__(self):
        """Initialize warehouse mapper."""
        # Create warehouse environment
        self.env = WarehouseEnvironment(gui=True, warehouse_size=(10, 10))
        
        # Spawn robot
        self.robot_id = self.env.spawn_robot(robot_type="husky", position=[-4, -4, 0.1])
        
        # Spawn obstacles (boxes)
        self.env.spawn_random_boxes(num_boxes=6)
        
        # Create LiDAR sensor
        self.lidar = LiDARSensor(num_rays=360, max_range=10.0)
        
        # Create occupancy grid mapper
        self.mapper = OccupancyGridMapper(width=12.0, height=12.0, resolution=0.1)
        
        # Setup visualization
        self.setup_visualization()
        
        print("\nWarehouse Mapper initialized!")
        print("Controls: The robot will explore automatically")
        print("Watch the map build in real-time!\n")
    
    def setup_visualization(self):
        """Setup matplotlib figure for real-time map visualization."""
        plt.ion()  # Interactive mode
        self.fig, self.axes = plt.subplots(1, 2, figsize=(14, 6))
        
        # Left plot: Occupancy grid
        self.ax_map = self.axes[0]
        self.ax_map.set_title('Occupancy Grid Map', fontsize=14, fontweight='bold')
        self.ax_map.set_xlabel('X Position (m)')
        self.ax_map.set_ylabel('Y Position (m)')
        self.ax_map.set_aspect('equal')
        
        # Right plot: Statistics
        self.ax_stats = self.axes[1]
        self.ax_stats.set_title('Mapping Statistics', fontsize=14, fontweight='bold')
        self.ax_stats.axis('off')
        
        plt.tight_layout()
    
    def update_visualization(self, scan_count):
        """Update the visualization with current map."""
        # Get probability grid
        prob_grid = self.mapper.get_probability_grid()
        
        # Clear and redraw map
        self.ax_map.clear()
        self.ax_map.set_title('Occupancy Grid Map', fontsize=14, fontweight='bold')
        self.ax_map.set_xlabel('X Position (m)')
        self.ax_map.set_ylabel('Y Position (m)')
        
        # Display map (flip vertically for correct orientation)
        extent = [-self.mapper.origin_x, self.mapper.origin_x,
                 -self.mapper.origin_y, self.mapper.origin_y]
        
        im = self.ax_map.imshow(prob_grid, cmap='gray_r', origin='lower',
                                extent=extent, vmin=0, vmax=1)
        
        # Add colorbar
        if not hasattr(self, 'colorbar'):
            self.colorbar = plt.colorbar(im, ax=self.ax_map, fraction=0.046, pad=0.04)
            self.colorbar.set_label('Occupancy Probability', rotation=270, labelpad=15)
        
        # Get robot position
        state = self.env.get_robot_state()
        if state:
            rx, ry = state['position'][0], state['position'][1]
            self.ax_map.plot(rx, ry, 'ro', markersize=10, label='Robot')
            
            # Draw robot orientation
            yaw = state['euler'][2]
            arrow_len = 0.5
            self.ax_map.arrow(rx, ry, 
                            arrow_len * np.cos(yaw), 
                            arrow_len * np.sin(yaw),
                            head_width=0.2, head_length=0.15, fc='r', ec='r')
        
        self.ax_map.legend()
        self.ax_map.grid(True, alpha=0.3)
        
        # Update statistics
        self.ax_stats.clear()
        self.ax_stats.axis('off')
        
        # Calculate statistics
        occupancy = self.mapper.get_occupancy_grid()
        total_cells = occupancy.size
        unknown = np.sum(occupancy == -1)
        free = np.sum(occupancy == 0)
        occupied = np.sum(occupancy == 1)
        
        unknown_pct = 100 * unknown / total_cells
        free_pct = 100 * free / total_cells
        occupied_pct = 100 * occupied / total_cells
        
        stats_text = f"""
        Mapping Progress
        ═══════════════════════════
        
        Scans Completed: {scan_count}
        
        Map Coverage:
        ─────────────────────────
        Unknown:   {unknown_pct:5.1f}%  ({unknown:6d} cells)
        Free:      {free_pct:5.1f}%  ({free:6d} cells)
        Occupied:  {occupied_pct:5.1f}%  ({occupied:6d} cells)
        
        Total Cells: {total_cells:,}
        Resolution: {self.mapper.resolution} m/cell
        
        Map Size: {self.mapper.width}m × {self.mapper.height}m
        """
        
        self.ax_stats.text(0.1, 0.5, stats_text, fontsize=11, 
                          verticalalignment='center', family='monospace')
        
        plt.pause(0.01)
    
    def explore_pattern(self, duration=60.0):
        """
        Make robot explore the warehouse in a pattern.
        
        Args:
            duration (float): Exploration duration in seconds
        """
        print("Starting exploration...")
        
        start_time = time.time()
        scan_count = 0
        last_scan_time = 0
        scan_interval = 0.5  # Scan every 0.5 seconds
        
        # Exploration pattern: move forward, turn, repeat
        phase = 0
        phase_start = time.time()
        phase_duration = 3.0  # Each phase lasts 3 seconds
        
        while time.time() - start_time < duration:
            current_time = time.time()
            
            # Control robot based on phase
            if phase == 0:  # Move forward
                self.env.move_robot(linear_velocity=0.5, angular_velocity=0.0)
                if current_time - phase_start > phase_duration:
                    phase = 1
                    phase_start = current_time
            elif phase == 1:  # Turn
                self.env.move_robot(linear_velocity=0.0, angular_velocity=0.8)
                if current_time - phase_start > 2.0:  # Turn for 2 seconds
                    phase = 0
                    phase_start = current_time
            
            # Step simulation
            self.env.step(sleep_time=1./240.)
            
            # Perform LiDAR scan periodically
            if current_time - last_scan_time >= scan_interval:
                # Get robot state
                state = self.env.get_robot_state()
                robot_x = state['position'][0]
                robot_y = state['position'][1]
                robot_yaw = state['euler'][2]
                
                # Perform scan
                ranges, sensor_angles = self.lidar.scan(self.robot_id)
                
                # Convert sensor angles to global angles
                global_angles = sensor_angles + robot_yaw
                
                # Update map
                self.mapper.update_from_scan(robot_x, robot_y, ranges, global_angles)
                
                scan_count += 1
                last_scan_time = current_time
                
                # Update visualization
                if scan_count % 2 == 0:  # Update every 2 scans
                    self.update_visualization(scan_count)
                
                print(f"Scan {scan_count}: Robot at ({robot_x:.2f}, {robot_y:.2f})")
        
        # Stop robot
        self.env.move_robot(0, 0)
        
        print(f"\nExploration complete! Total scans: {scan_count}")
    
    def save_map(self, filename='warehouse_map.png'):
        """Save the final map to file."""
        prob_grid = self.mapper.get_probability_grid()
        
        plt.figure(figsize=(10, 10))
        plt.imshow(prob_grid, cmap='gray_r', origin='lower')
        plt.title('Final Warehouse Occupancy Grid Map', fontsize=16, fontweight='bold')
        plt.xlabel('Grid X')
        plt.ylabel('Grid Y')
        plt.colorbar(label='Occupancy Probability')
        plt.tight_layout()
        plt.savefig(filename, dpi=150)
        print(f"Map saved to {filename}")
    
    def run(self):
        """Run the complete mapping demo."""
        try:
            # Explore warehouse
            self.explore_pattern(duration=30.0)
            
            # Final visualization update
            self.update_visualization(scan_count=999)
            
            # Save map
            self.save_map('activity3_warehouse_map.png')
            
            print("\n" + "="*50)
            print("Mapping demo complete!")
            print("="*50)
            print("\nPress Enter to close...")
            input()
            
        finally:
            plt.ioff()
            plt.close('all')
            self.env.close()


# Main execution
if __name__ == "__main__":
    print("="*60)
    print("Activity 3: Real-Time Warehouse Mapping")
    print("="*60)
    print("\nThis demo shows:")
    print("  1. LiDAR sensor simulation in PyBullet")
    print("  2. Real-time occupancy grid mapping")
    print("  3. Robot exploration in warehouse environment")
    print("\nThe robot will explore for 30 seconds.")
    print("Watch the map build in real-time!")
    print("="*60)
    print()
    
    # Create and run mapper
    mapper = WarehouseMapper()
    mapper.run()



---

## Summary & Next Steps

Congratulations on completing this lab! Before moving on:

1. **Review** your outputs and make sure they match expected behavior
2. **Experiment** by modifying parameters and observing the effects
3. **Reflect** on what you learned — write a brief paragraph in your report

---

*Dr. Akram Fatayer · [a.fatayer@zuj.edu.jo](mailto:a.fatayer@zuj.edu.jo) · [LinkedIn](https://www.linkedin.com/in/akram-fatayer/) · Al-Zaytoonah University of Jordan*
